# Evaluate LAMPS Full Pipeline trên D2 (paper §5.2 / §5.3)

Yêu cầu trên Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/d2/files.jsonl`
- `NT230/data/d2/packages.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys, json
from pathlib import Path
from collections import defaultdict

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_D2 = '/content/drive/My Drive/NT230/data/d2'

!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

# model.bin từ d1
os.makedirs('/content/saved_models/checkpoint-best-acc', exist_ok=True)
shutil.copy(f'{DRIVE_D1}/saved_models/checkpoint-best-acc/model.bin',
            '/content/saved_models/checkpoint-best-acc/model.bin')
print('✅ model.bin:', round(os.path.getsize('/content/saved_models/checkpoint-best-acc/model.bin')/1e6), 'MB')

# D2 data
shutil.copy(f'{DRIVE_D2}/files.jsonl',    '/content/d2_files.jsonl')
shutil.copy(f'{DRIVE_D2}/packages.jsonl', '/content/d2_packages.jsonl')
print('✅ d2_files.jsonl:   ', sum(1 for _ in open('/content/d2_files.jsonl')), 'records')
print('✅ d2_packages.jsonl:', sum(1 for _ in open('/content/d2_packages.jsonl')), 'records')

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU')

In [ ]:
from lamps.agents.classifier import ClassifierAgent, FileClassification
from lamps.agents.extractor import ExtractedFile
from lamps.agents.verdict import VerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

classifier    = ClassifierAgent(checkpoint='/content/saved_models/checkpoint-best-acc/model.bin', batch_size=64)
verdict_agent = VerdictAgent(llm=None)
print('✅ Agents loaded')

In [ ]:
file_records    = list(read_jsonl(Path('/content/d2_files.jsonl')))
package_records = list(read_jsonl(Path('/content/d2_packages.jsonl')))
print(f'Files: {len(file_records)} | Packages: {len(package_records)}')

In [ ]:
# Extractor Agent — rule-based filter
NOISY = {'tests','test','testing','docs','doc','examples','_vendor','vendor'}
def is_relevant(path):
    parts = str(path).lower().replace('\\','/').split('/')
    return not any(p in NOISY for p in parts) and not parts[-1].startswith('test_')

filtered = [r for r in file_records if is_relevant(r.get('path', ''))]
print(f'After filter: {len(filtered)} / {len(file_records)} files')

In [ ]:
# Classifier Agent
files = [ExtractedFile(package=str(r['package']), path=Path('<memory>'),
                       rel_path=str(r.get('path','')), source=str(r['func']))
         for r in filtered]

print(f'Classifying {len(files)} files...')
classifications = classifier.classify_files(files)

y_file_true = [int(r['target']) for r in filtered]
y_file_pred = [c.target for c in classifications]
print('\n=== File-level ===')
print(format_report(classification_report(y_file_true, y_file_pred)))

In [ ]:
# Verdict Agent — conservative aggregation
cls_by_pkg = defaultdict(list)
for cls in classifications:
    cls_by_pkg[cls.package].append(cls)

y_pkg_true, y_pkg_pred, pkg_preds = [], [], []
for pkg in package_records:
    verdict = verdict_agent.aggregate(pkg['package'], cls_by_pkg.get(pkg['package'], []))
    y_pkg_true.append(int(pkg['label']))
    y_pkg_pred.append(verdict.target)
    pkg_preds.append({'package': pkg['package'], 'target': int(pkg['label']),
                      'predicted': verdict.target, 'n_malicious_files': len(verdict.malicious_files)})

pkg_report = classification_report(y_pkg_true, y_pkg_pred)
print('\n=== Package-level (paper Table 3) ===')
print(format_report(pkg_report))

In [ ]:
# Save về Drive
os.makedirs('/content/results_d2', exist_ok=True)
with open('/content/results_d2/package_report.json','w') as f:
    json.dump(pkg_report.to_dict(), f, indent=2)
with open('/content/results_d2/package_predictions.jsonl','w') as f:
    f.write('\n'.join(json.dumps(p) for p in pkg_preds))

shutil.copytree('/content/results_d2', f'{DRIVE_D2}/results', dirs_exist_ok=True)
print(f'✅ Saved to Drive: NT230/data/d2/results/')